# Train, tune, and deploy a custom ML model using RigoBERTa Clinical Classification Algorithm from AWS Marketplace 


RigoBERTa Clinical Classification is a fine-tunable state-of-the-art clinical encoder language model for Spanish text classification.

It was built by further pretraining our general-purpose RigoBERTa 2 model on a meticulously Spanish curated clinical corpus, significantly improving performance on multiple clinical NLP benchmarks while offering robust language understanding in the clinical domain.

This sample notebook shows you how to train a custom ML model using [RigoBERTa-Clinical Classification](https://aws.amazon.com/marketplace/seller-profile?id=seller-gjhvwl4dgbrqk) from AWS Marketplace.

## Pre-requisites
1. **Note**: This notebook contains elements which render correctly in Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that IAM role used has **AmazonSageMakerFullAccess**
1. Some hands-on experience using [Amazon SageMaker](https://aws.amazon.com/sagemaker/).
1. To use this algorithm successfully, ensure that:
    1. Either your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used: 
        1. **aws-marketplace:ViewSubscriptions**
        1. **aws-marketplace:Unsubscribe**
        1. **aws-marketplace:Subscribe**  
    2. or your AWS account has a subscription to [RigoBERTa-Clinical Classification](https://aws.amazon.com/marketplace/seller-profile?id=seller-gjhvwl4dgbrqk). 

## Contents
1. [Subscribe to the algorithm](#1.-Subscribe-to-the-algorithm)
1. [Prepare dataset](#2.-Prepare-dataset)
	1. [Dataset format expected by the algorithm](#A.-Dataset-format-expected-by-the-algorithm)
	1. [Configure and visualize train and test dataset](#B.-Configure-and-visualize-train-and-test-dataset)
	1. [Upload datasets to Amazon S3](#C.-Upload-datasets-to-Amazon-S3)
1. [Train a machine learning model](#3:-Train-a-machine-learning-model)
	1. [Set up environment](#3.1-Set-up-environment)
	1. [Train a model](#3.2-Train-a-model)
1. [Deploy model and verify results](#4:-Deploy-model-and-verify-results)
    1. [Deploy trained model](#A.-Deploy-trained-model)
    1. [Create input payload](#B.-Create-input-payload)
    1. [Perform real-time inference](#C.-Perform-real-time-inference)
    1. [Visualize output](#D.-Visualize-output)
    1. [Calculate relevant metrics](#E.-Calculate-relevant-metrics)
    1. [Delete the endpoint](#F.-Delete-the-endpoint)
1. [Tune your model (optional)](#5:-Tune-your-model-(optional))
	1. [Tuning Guidelines](#A.-Tuning-Guidelines)
	1. [Define Tuning configuration](#B.-Define-Tuning-configuration)
	1. [Run a model tuning job](#C.-Run-a-model-tuning-job)
1. [Perform Batch inference](#6.-Perform-Batch-inference)
1. [Clean-up](#7.-Clean-up)
	1. [Delete the model](#A.-Delete-the-model)
	1. [Unsubscribe to the listing (optional)](#B.-Unsubscribe-to-the-listing-(optional))


## Usage instructions
You can run this notebook one cell at a time (By using Shift+Enter for running a cell).

## 1. Subscribe to the algorithm

To subscribe to the algorithm:
1. Open the algorithm listing page [RigoBERTa-Clinical Classification](https://aws.amazon.com/marketplace/seller-profile?id=seller-gjhvwl4dgbrqk).
1. On the AWS Marketplace listing,  click on **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you agree with EULA, pricing, and support terms. 
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn**. This is the algorithm ARN that you need to specify while training a custom ML model. Copy the ARN corresponding to your region and specify the same in the following cell.

In [ ]:
algo_arn = "arn:aws:sagemaker:eu-west-1:459676377564:algorithm/RigoBERTa-Clinical-Classification"

## 2. Prepare dataset

In [ ]:
import json
import sagemaker as sage

### A. Dataset format expected by the algorithm

Rigoberta-Clinical Classification handles both JSON and CSV training datasets in the following format
```json
[
    {
        "text": "<text1>",
        "label": "<label1>"
    },
    {
        "text": "<text2>",
        "label": "<label2>"
    },
    ...
]
```

You can also find more information about dataset format in **Usage Information** section of [RigoBERTa-Clinical Classification](https://aws.amazon.com/marketplace/)

### B. Configure and visualize train and test dataset

In [ ]:
training_dataset = "data/train/classification_train.json"

In [ ]:
with open(training_dataset, "r") as f:
    training_data = json.load(f)

for row in training_data:
    print(row)

### C. Upload datasets to Amazon S3

In [ ]:
sagemaker_session = sage.Session()
bucket = sagemaker_session.default_bucket()
bucket

In [ ]:
s3_training_dataset = sagemaker_session.upload_data(
    training_dataset, bucket=bucket, key_prefix="rigoberta-clinical-classification"
)

## 3: Train a machine learning model

Now that dataset is available in an accessible Amazon S3 bucket, we are ready to train a machine learning model. 

### 3.1 Set up environment

In [ ]:
role = sage.get_execution_role()

In [ ]:
output_location = "s3://{}/rigoberta-clinical-classification/{}".format(
    bucket, "output"
)

### 3.2 Train a model

You can also find more information about dataset format in **Hyperparameters** section of [RigoBERTa-Clinical Classification](https://aws.amazon.com/marketplace/).

In [ ]:
# Define hyperparameters
hyperparameters = {
    'epochs': 1,
    'batch_size': 32,
    'learning_rate': 1e-5,
}

For information on creating an `Estimator` object, see [documentation](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html)

In [ ]:
# Create an estimator object for running a training job
estimator = sage.algorithm.AlgorithmEstimator(
    algorithm_arn=algo_arn,
    base_job_name="rigoberta-clinical-classification",
    role=role,
    instance_count=1,
    instance_type="ml.g5.4xlarge",
    input_mode="File",
    output_path=output_location,
    sagemaker_session=sagemaker_session,
    hyperparameters=hyperparameters,
)
# Run the training job.
estimator.fit({"train": s3_training_dataset})

In [ ]:
 # Captured metrics can be accessed
sage.TrainingJobAnalytics(
    training_job_name=estimator.latest_training_job.name
).dataframe()

See this [blog-post](https://aws.amazon.com/blogs/machine-learning/easily-monitor-and-visualize-metrics-while-training-models-on-amazon-sagemaker/) for more information how to visualize metrics during the process. You can also open the training job from [Amazon SageMaker console](https://console.aws.amazon.com/sagemaker/home?#/jobs/) and monitor the metrics/logs in **Monitor** section.

## 4: Deploy model and verify results

Now you can deploy the model for performing real-time inference.

In [ ]:
model_name = "RigoBERTa-Clinical Classification"

content_type = "application/json"

real_time_inference_instance_type = (
    "ml.g5.4xlarge"
)
batch_transform_inference_instance_type = (
    "ml.g5.4xlarge"
)

### A. Deploy trained model

In [ ]:
predictor = estimator.deploy(
    1, real_time_inference_instance_type, serializer=sage.base_serializers.JSONSerializer(),
    deserializer=sage.base_deserializers.JSONDeserializer(),
    container_startup_health_check_timeout=60*5,
)

Once endpoint is created, you can perform real-time inference.

### B. Create input payload

In [ ]:
input_dataset = "data/input/classification_input.json"

with open(input_dataset, "r") as f:
    input_data = json.load(f)

print(input_data)

<Add code snippet that shows the payload contents>

### C. Perform real-time inference

In [ ]:
response = predictor.predict(input_data)

### D. Visualize output

In [ ]:
for row in response:
    print(row)

If [Amazon SageMaker Model Monitor](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor.html) supports the type of problem you are trying to solve using this algorithm, use the following examples to add Model Monitor support to your product:
For sample code to enable and monitor the model, see following notebooks:
1. [Enable Amazon SageMaker Model Monitor](https://github.com/awslabs/amazon-sagemaker-examples/blob/master/sagemaker_model_monitor/enable_model_monitor/SageMaker-Enable-Model-Monitor.ipynb)
2. [Amazon SageMaker Model Monitor - visualizing monitoring results](https://github.com/awslabs/amazon-sagemaker-examples/blob/master/sagemaker_model_monitor/visualization/SageMaker-Model-Monitor-Visualize.ipynb)

### F. Delete the endpoint

Now that you have successfully performed a real-time inference, you do not need the endpoint any more. you can terminate the same to avoid being charged.

In [ ]:
predictor.delete_endpoint(delete_endpoint_config=True)

Since this is an experiment, you do not need to run a hyperparameter tuning job. However, if you would like to see how to tune a model trained using a third-party algorithm with Amazon SageMaker's hyperparameter tuning functionality, you can run the optional tuning step.

## 6. Perform Batch inference

In this section, you will perform batch inference using multiple input payloads together.

In [ ]:
# upload the batch-transform job input files to S3
transform_input_folder = "data/input"
s3prefix_transform_input_folder = f"{model_name}/{transform_input_folder}"
transform_input = sagemaker_session.upload_data(transform_input_folder, key_prefix=s3prefix_transform_input_folder)
print("Transform input uploaded to " + transform_input)

In [ ]:
# Run the batch-transform job
transformer = estimator.transformer(1, batch_transform_inference_instance_type)
transformer.transform(transform_input, content_type=content_type)
transformer.wait()

In [ ]:
# output is available on following path
transformer.output_path

In [ ]:
output_path = "output"
!aws s3 cp $transformer.output_path $output_path --recursive

In [ ]:
with open(f"{output_path}/classification_input.json.out", "r") as f:
    response = json.load(f)

for row in response:
    print(row)

## 7. Clean-up

### A. Delete the model

In [ ]:
predictor.delete_model()

### B. Unsubscribe to the listing (optional)

If you would like to unsubscribe to the algorithm, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note - You can find this information by looking at the container name associated with the model. 

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__  to cancel the subscription.



## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|Algorithm|Sample_Notebook_Template|title_of_your_product-Algorithm.ipynb)
